In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("loan_approval_data (2).csv")


In [ ]:
df.head()
df.info()


# HANDLING MISSING VALUES

In [ ]:
categorical_col = df.select_dtypes(include = ["object"]).columns
numerical_col = df.select_dtypes(include = ["float64"]).columns

In [ ]:
categorical_col

In [ ]:
from sklearn.impute import SimpleImputer
num_imp = SimpleImputer(strategy = "mean")
df[numerical_col] = num_imp.fit_transform(df[numerical_col])

In [ ]:
df.head()

In [ ]:
cat_imp = SimpleImputer(strategy = "most_frequent")
df[categorical_col] = cat_imp.fit_transform(df[categorical_col])

In [ ]:
df.head()

# EDA - exploratory data analysis (patterns, relations etc)

In [ ]:
# how balanced our classes are?

classes_count = df["Loan_Approved"].value_counts()
plt.pie(classes_count , labels = ["No" , "yes"] , autopct = "%1.1f%%")
plt.title("Is loan approved or not")

In [ ]:
#analyze categories
gender_based = df["Gender"].value_counts()
# plt.pie(gender_based ,labels =["male" , "female"] , autopct = "%1.1f%%")
# plt.title("based on gender loan_approved")
ax = sns.barplot(gender_based)
ax.bar_label(ax.containers[0])

In [ ]:
# analyze income

sns.histplot(
    data = df,
    x = "Coapplicant_Income",
    bins = 20
)

In [ ]:
sns.histplot(
    data = df,
    x = "Applicant_Income",
    bins = 20
)

In [ ]:
 # outliers - box plot
fig , axes = plt.subplots(2,2)

sns.boxplot(ax = axes[0,0],data = df, x = "Loan_Approved", y = "Applicant_Income")
sns.boxplot(ax = axes[0,1],data = df, x = "Loan_Approved", y = "Credit_Score")
sns.boxplot(ax = axes[1,0],data = df, x = "Loan_Approved", y = "DTI_Ratio")
sns.boxplot(ax = axes[1,1],data = df, x = "Loan_Approved", y = "Savings")

plt.tight_layout()

In [ ]:
sns.histplot(
    data = df,
    x = "Credit_Score",
    hue = "Loan_Approved",
    bins = 20,
    multiple ="dodge"
)

In [ ]:
# removing applicant id
df = df.drop("Applicant_ID" , axis = 1)


In [ ]:
df.info()

# ENCODING 


In [ ]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder

In [ ]:
le = LabelEncoder()
df["Education_Level"]  = le.fit_transform(df["Education_Level"])
df["Loan_Approved"]  = le.fit_transform(df["Loan_Approved"])

df.head()


In [ ]:
cols = [ "Employment_Status" , "Marital_Status" ,"Gender" , "Employer_Category" , "Loan_Approved" , "Property_Area" , "Loan_Purpose"]

ohe = OneHotEncoder( drop = "first" , sparse_output= False ,handle_unknown="ignore" )
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded , columns = ohe.get_feature_names_out(cols) , index = df.index)
df = pd.concat([df.drop(columns = cols) , encoded_df] , axis =1)

In [ ]:
df.head()

# corelation heatmap

In [ ]:
num_cols = df.select_dtypes(include = "number")
corr_matrix = num_cols.corr()
plt.figure(figsize =(15,8))
sns.heatmap(
    corr_matrix ,
    annot=True,
    fmt =".2f",
    cmap = "coolwarm"
)

In [ ]:
num_cols.corr()["Loan_Approved_1"].sort_values(ascending = False)
3
6+6 

# TRAIN-TEST-SPLIT + FEATURE SCALING

In [ ]:
x = df.drop("Loan_Approved_1" , axis = 1)
y = df["Loan_Approved_1"]

In [ ]:
y.head()

In [ ]:
x_train , x_test, y_train ,y_test = train_test_split( x ,y , random_state = 42, test_size =0.2)

In [ ]:
x_train.head()
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# TRAIN & EVALUATE MODELS

In [ ]:
# logisic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
log_model = LogisticRegression()
log_model.fit(x_train_scaled,y_train)


In [ ]:
print("LogisticRegression")
y_pred = log_model.predict(x_test_scaled)
print("precision ", precision_score(y_test,y_pred))
print("acccuracy ", accuracy_score(y_test,y_pred))
print("recall ", recall_score(y_test,y_pred))
print("f1  ", f1_score(y_test,y_pred))
print("confusion_matrix ", confusion_matrix(y_test,y_pred))


In [ ]:
# for knn
print("KNN MODEL")
from sklearn.neighbors import KNeighborsClassifier
k_model = KNeighborsClassifier(n_neighbors = 5)
k_model.fit(x_train_scaled,y_train)
y_pred = k_model.predict(x_test_scaled)
print("precision ", precision_score(y_test,y_pred))
print("acccuracy ", accuracy_score(y_test,y_pred))
print("recall ", recall_score(y_test,y_pred))
print("f1  ", f1_score(y_test,y_pred))
print("confusion_matrix ", confusion_matrix(y_test,y_pred))

In [ ]:
# neive bayes
from sklearn.naive_bayes import GaussianNB
neive_model = GaussianNB()
print("neive bayes MODEL")
neive_model.fit(x_train_scaled,y_train)
y_pred =neive_model.predict(x_test_scaled)
print("precision ", precision_score(y_test,y_pred))
print("acccuracy ", accuracy_score(y_test,y_pred))
print("recall ", recall_score(y_test,y_pred))
print("f1  ", f1_score(y_test,y_pred))
print("confusion_matrix ", confusion_matrix(y_test,y_pred))

 # based on precision score neive bayes is best

# feature engineering

In [ ]:
# ADD AND TRANSFORM FEATURES

df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
df["Credit_Score_sq"] = df["Credit_Score"] ** 2

# df["Applicant_Income_log"] = np.log1p(df["Applicant_Income"])

x = df.drop(columns = ["Loan_Approved_1","Credit_Score" ,"DTI_Ratio", "Applicant_Income"])
y = df["Loan_Approved_1"]

#train _test _split
x_train , x_test, y_train ,y_test = train_test_split( x ,y , random_state = 42, test_size =0.2)
# scaling
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:

print("LogisticRegression")
y_pred = log_model.predict(x_test_scaled)
print("precision ", precision_score(y_test,y_pred))
print("acccuracy ", accuracy_score(y_test,y_pred))
print("recall ", recall_score(y_test,y_pred))
print("f1  ", f1_score(y_test,y_pred))
print("confusion_matrix ", confusion_matrix(y_test,y_pred))

In [ ]:
print("KNN MODEL")
from sklearn.neighbors import KNeighborsClassifier
k_model = KNeighborsClassifier(n_neighbors = 5)
k_model.fit(x_train_scaled,y_train)
y_pred = k_model.predict(x_test_scaled)
print("precision ", precision_score(y_test,y_pred))
print("acccuracy ", accuracy_score(y_test,y_pred))
print("recall ", recall_score(y_test,y_pred))
print("f1  ", f1_score(y_test,y_pred))
print("confusion_matrix ", confusion_matrix(y_test,y_pred))

In [ ]:
from sklearn.naive_bayes import GaussianNB
neive_model = GaussianNB()
print("neive bayes MODEL")
neive_model.fit(x_train_scaled,y_train)
y_pred =neive_model.predict(x_test_scaled)
print("precision ", precision_score(y_test,y_pred))
print("acccuracy ", accuracy_score(y_test,y_pred))
print("recall ", recall_score(y_test,y_pred))
print("f1  ", f1_score(y_test,y_pred))
print("confusion_matrix ", confusion_matrix(y_test,y_pred))

In [ ]:
import streamlit as st
import joblib
import pandas as pd

# Model load
model = joblib.load("model.pkl")

st.title("Loan Approval Prediction")

# User Input
income = st.number_input("Applicant Income")
loan_amount = st.number_input("Loan Amount")
credit_score = st.number_input("Credit Score")

if st.button("Predict"):
    
    input_data = pd.DataFrame({
        "Applicant_Income": [income],
        "LoanAmount": [loan_amount],
        "Credit_Score": [credit_score]
    })
    
    prediction = model.predict(input_data)

    if prediction[0] == 1:
        st.success("Loan Approved ✅")
    else:
        st.error("Loan Not Approved ❌")

In [ ]:
import joblib

joblib.dump(neive_model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")


In [ ]:
import streamlit as st

In [ ]:
print(x.columns.tolist())

In [ ]:
joblib.dump(x.columns.tolist(), "features.pkl")

In [ ]:
import streamlit as st
import pandas as pd

import joblib

# Load model, scaler and feature names
model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")
features = joblib.load("features.pkl")

# Title
st.title("🏦 Loan Approval Prediction")
st.write("Enter your details below")

# Numerical Inputs
applicant_income = st.number_input("Applicant Income", min_value=0.0)
coapplicant_income = st.number_input("Coapplicant Income", min_value=0.0)

age = st.number_input("Age", min_value=18, max_value=100)

dependents = st.number_input("Dependents", min_value=0)

credit_score = st.number_input(
    "Credit Score",
    min_value=0.0,
    max_value=1000.0
)

existing_loans = st.number_input("Existing Loans", min_value=0)

dti_ratio = st.number_input(
    "DTI Ratio",
    min_value=0.0
)

savings = st.number_input("Savings", min_value=0.0)

collateral_value = st.number_input(
    "Collateral Value",
    min_value=0.0
)

loan_amount = st.number_input(
    "Loan Amount",
    min_value=0.0
)

loan_term = st.number_input(
    "Loan Term",
    min_value=0
)

# Categorical Inputs
employment_status = st.selectbox(
    "Employment Status",
    ["Salaried", "Self-employed", "Unemployed"]
)

marital_status = st.selectbox(
    "Marital Status",
    ["Married", "Single"]
)

gender = st.selectbox(
    "Gender",
    ["Female", "Male"]
)

employer_category = st.selectbox(
    "Employer Category",
    ["Government", "MNC", "Private", "Unemployed"]
)

property_area = st.selectbox(
    "Property Area",
    ["Rural", "Semiurban", "Urban"]
)

loan_purpose = st.selectbox(
    "Loan Purpose",
    ["Car", "Education", "Home", "Personal"]
)

# Prediction Button
if st.button("Predict Loan Approval"):

    # Create dataframe with all features as 0
    input_data = pd.DataFrame(
        [[0] * len(features)],
        columns=features
    )

    # Fill numerical columns
    values = {
        "Applicant_Income": applicant_income,
        "Coapplicant_Income": coapplicant_income,
        "Age": age,
        "Dependents": dependents,
        "Credit_Score": credit_score,
        "Existing_Loans": existing_loans,
        "DTI_Ratio": dti_ratio,
        "Savings": savings,
        "Collateral_Value": collateral_value,
        "Loan_Amount": loan_amount,
        "Loan_Term": loan_term,

        # Engineered features
        "DTI_Ratio_sq": dti_ratio ** 2,
        "Credit_Score_sq": credit_score ** 2
    }

    # Put values only if column exists
    for col, value in values.items():
        if col in input_data.columns:
            input_data[col] = value

    # One-hot encoded categorical values
    categorical_columns = [
        f"Employment_Status_{employment_status}",
        f"Marital_Status_{marital_status}",
        f"Gender_{gender}",
        f"Employer_Category_{employer_category}",
        f"Property_Area_{property_area}",
        f"Loan_Purpose_{loan_purpose}"
    ]

    for col in categorical_columns:
        if col in input_data.columns:
            input_data[col] = 1

    # Scale the data
    input_scaled = scaler.transform(input_data)

    # Prediction
    prediction = model.predict(input_scaled)

    if prediction[0] == 1:
        st.success("🎉 Loan Approved!")
    else:
        st.error("❌ Loan Not Approved")



        

In [ ]:
%%writefile app.py
import streamlit as st
import pickle
import numpy as np
import pandas as pd

# Load saved models and preprocessing objects
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

st.set_page_config(page_title="Credit Wise Loan System", layout="centered")

st.title("Credit Wise Loan Approval System")
st.write("Enter details below to predict loan approval status.")

# Input fields
applicant_income = st.number_input("Applicant Income", min_value=0.0, value=50000.0)
credit_score = st.number_input("Credit Score", min_value=0.0, max_value=900.0, value=700.0)
dti_ratio = st.number_input("DTI Ratio", min_value=0.0, max_value=1.0, value=0.3)

if st.button("Predict Loan Status"):
    # Feature transformations
    dti_ratio_sq = dti_ratio ** 2
    credit_score_sq = credit_score ** 2

    # DataFrame creation
    input_data = pd.DataFrame([{
        'DTI_Ratio_sq': dti_ratio_sq,
        'Credit_Score_sq': credit_score_sq
    }])

    # Scale and Predict
    scaled_data = scaler.transform(input_data)
    prediction = model.predict(scaled_data)

    # Output Condition
    if prediction[0] == 1:
        st.success("Loan Status: Approved!")
    else:
        st.error("Loan Status: Rejected!")

In [ ]:
import pickle

# 1. Scaler save karein
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# 2. Model save karein (agar aapne neive_model banaya hai)
with open('model.pkl', 'wb') as f:
    pickle.dump(neive_model, f)

In [ ]:
print(scaler.feature_names_in_)

In [ ]:
%%writefile app.py
import streamlit as st
import pickle
import numpy as np
import pandas as pd

# Load saved model and scaler
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

st.set_page_config(page_title="Credit Wise Loan System", layout="wide")

st.title("💳 Credit Wise Loan Approval System")
st.write("Fill in the applicant details below to check loan eligibility.")

# Layout in Columns
col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("Personal Details")
    age = st.number_input("Age", min_value=18, max_value=100, value=30)
    gender = st.selectbox("Gender", ["Male", "Female"])
    marital_status = st.selectbox("Marital Status", ["Single", "Married"])
    dependents = st.number_input("Dependents", min_value=0, max_value=10, value=0)
    education_level = st.selectbox("Education Level", ["Undergraduate", "Graduate / Higher"])

with col2:
    st.subheader("Financial & Employment")
    coapplicant_income = st.number_input("Coapplicant Income", min_value=0.0, value=0.0)
    savings = st.number_input("Savings", min_value=0.0, value=50000.0)
    collateral_value = st.number_input("Collateral Value", min_value=0.0, value=100000.0)
    employment_status = st.selectbox("Employment Status", ["Salaried", "Self-employed", "Unemployed"])
    employer_category = st.selectbox("Employer Category", ["Government", "MNC", "Private", "Unemployed"])

with col3:
    st.subheader("Loan Details")
    loan_amount = st.number_input("Loan Amount", min_value=1000.0, value=200000.0)
    loan_term = st.number_input("Loan Term (Months)", min_value=6, max_value=360, value=36)
    existing_loans = st.number_input("Existing Loans Count", min_value=0, max_value=10, value=0)
    property_area = st.selectbox("Property Area", ["Semiurban", "Urban", "Rural"])
    loan_purpose = st.selectbox("Loan Purpose", ["Car", "Education", "Home", "Personal", "Other"])
    credit_score = st.number_input("Credit Score", min_value=300.0, max_value=900.0, value=750.0)
    dti_ratio = st.number_input("DTI Ratio", min_value=0.0, max_value=1.0, value=0.3)

st.markdown("---")

if st.button("Predict Loan Status", use_container_width=True):
    # Engineered Features
    dti_ratio_sq = dti_ratio ** 2
    credit_score_sq = credit_score ** 2

    # Categorical One-Hot Conversions
    education_val = 1 if education_level == "Graduate / Higher" else 0
    emp_salaried = 1 if employment_status == "Salaried" else 0
    emp_self = 1 if employment_status == "Self-employed" else 0
    emp_unemployed = 1 if employment_status == "Unemployed" else 0

    marital_single = 1 if marital_status == "Single" else 0
    gender_male = 1 if gender == "Male" else 0

    emp_cat_gov = 1 if employer_category == "Government" else 0
    emp_cat_mnc = 1 if employer_category == "MNC" else 0
    emp_cat_pvt = 1 if employer_category == "Private" else 0
    emp_cat_unemp = 1 if employer_category == "Unemployed" else 0

    prop_semiurban = 1 if property_area == "Semiurban" else 0
    prop_urban = 1 if property_area == "Urban" else 0

    purpose_car = 1 if loan_purpose == "Car" else 0
    purpose_edu = 1 if loan_purpose == "Education" else 0
    purpose_home = 1 if loan_purpose == "Home" else 0
    purpose_personal = 1 if loan_purpose == "Personal" else 0

    # Build DataFrame matching exact feature list and column order
    input_df = pd.DataFrame([{
        'Coapplicant_Income': coapplicant_income,
        'Age': age,
        'Dependents': dependents,
        'Existing_Loans': existing_loans,
        'Savings': savings,
        'Collateral_Value': collateral_value,
        'Loan_Amount': loan_amount,
        'Loan_Term': loan_term,
        'Education_Level': education_val,
        'Employment_Status_Salaried': emp_salaried,
        'Employment_Status_Self-employed': emp_self,
        'Employment_Status_Unemployed': emp_unemployed,
        'Marital_Status_Single': marital_single,
        'Gender_Male': gender_male,
        'Employer_Category_Government': emp_cat_gov,
        'Employer_Category_MNC': emp_cat_mnc,
        'Employer_Category_Private': emp_cat_pvt,
        'Employer_Category_Unemployed': emp_cat_unemp,
        'Property_Area_Semiurban': prop_semiurban,
        'Property_Area_Urban': prop_urban,
        'Loan_Purpose_Car': purpose_car,
        'Loan_Purpose_Education': purpose_edu,
        'Loan_Purpose_Home': purpose_home,
        'Loan_Purpose_Personal': purpose_personal,
        'DTI_Ratio_sq': dti_ratio_sq,
        'Credit_Score_sq': credit_score_sq
    }])

    # Transform & Predict
    scaled_data = scaler.transform(input_df)
    prediction = model.predict(scaled_data)

    st.subheader("Result:")
    if prediction[0] == 1:
        st.success("🎉 Congratulations! The Loan application is APPROVED.")
    else:
        st.error("❌ We regret to inform you that the Loan application is REJECTED.")

In [ ]:
import pickle
import os

# 1. Naya folder create karein (agar pehle se nahi hai)
folder_name = "saved_models"
os.makedirs(folder_name, exist_ok=True)

# 2. Model ko folder ke andar save (dump) karein
with open(os.path.join(folder_name, 'model.pkl'), 'wb') as f:
    pickle.dump(neive_model, f)  # ya log_model

# 3. Scaler ko folder ke andar save karein
with open(os.path.join(folder_name, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print(f"Model aur Scaler successfully '{folder_name}' folder me save ho gaye hain!")

In [ ]:
import os
import pickle
import streamlit as st

# Folder se model aur scaler load karein
model_path = os.path.join('saved_models', 'model.pkl')
scaler_path = os.path.join('saved_models', 'scaler.pkl')

with open(model_path, 'rb') as f:
    model = pickle.load(f)

with open(scaler_path, 'rb') as f:
    scaler = pickle.load(f)